# Colab Runner: PPO + Imitation Learning

Runs the full pipeline on Colab. Code is cloned from GitHub; trained models and results are persisted to Google Drive so disconnects do not lose work.

**Compute note:** PPO training is CPU-bound (MuJoCo physics), so a GPU runtime does not speed up the expert; it only helps from-scratch BC. Use a GPU runtime for the BC ablation/sweep, CPU is fine for PPO. Long runs need Colab Pro or an active tab to avoid idle disconnects.

## 1. Clone the code from GitHub

In [ ]:
REPO = 'https://github.com/em-ech/rl-ppo-imitation-learning.git'
BRANCH = 'extended-e1-e2-and-sac-bonus'  # set to 'main' once this is merged
CODE_DIR = '/content/GroupProject'
import os, sys, shutil
os.chdir('/content')  # never stand inside the dir we are about to delete
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
!git clone -q --branch {BRANCH} {REPO} {CODE_DIR}
assert os.path.isdir(CODE_DIR), 'clone failed'
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print('code in', CODE_DIR, '->', sorted(os.listdir(CODE_DIR))[:8])

## 2. Mount Drive for persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/rl_project'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 3. Install pinned dependencies

This pins numpy 1.26 (required by gymnasium 0.29, which imitation 1.0 forces), downgrading Colab's preinstalled numpy 2.x. **After this cell finishes you MUST restart the runtime** (Runtime -> Restart session) for the downgrade to take effect, then re-run cells 1-2 and continue from cell 4. Skip this cell on the second pass (packages persist across a restart). The long 'dependency resolver' conflict warnings about jax/transformers/opencv/torchvision are harmless: this project does not use those packages.

In [ ]:
!pip -q install -r /content/GroupProject/requirements.txt
print('\nInstall done. NOW: Runtime -> Restart session, then re-run '
      'cells 1-2 and continue from cell 4.')

## 4. Configure persistence + sanity check

Pointing `PROJECT_DATA_ROOT` at Drive sends models/, data/, outputs/, logs/ to Drive so they survive disconnects. `os.environ` changes are inherited by the `!python ...` calls below.

In [ ]:
import sys, os  # re-assert in case the runtime was restarted
sys.path.insert(0, '/content/GroupProject'); os.chdir('/content/GroupProject')
os.environ['PROJECT_DATA_ROOT'] = DRIVE_ROOT
from src import config
print('device:', config.device())
print('models ->', config.MODELS_DIR)
import gymnasium as gym
for e in ['Walker2d-v4', 'Ant-v4']:
    gym.make(e).reset(seed=0); print('ok', e)

## 5. Stage 1 - PPO experts (M1)

Walker2d with the validated config. For Ant, the faithful rl-zoo3 Optuna-tuned profile (`tuned_ant`, n_envs=1, ~1e7 steps) is the best shot at the 4000 target but is slow (roughly 18h on Colab). If the run is interrupted, re-run the same cell with `resume` appended: it continues from the last 100k checkpoint on Drive instead of restarting. Each run writes best_model (and vecnormalize.pkl for Ant) to Drive.

In [ ]:
!cd /content/GroupProject && python train_expert.py Walker2d-v4 4000000 4

In [ ]:
# Faithful rl-zoo3 tuned Ant (slow). Append 'resume' to continue after a disconnect.
!cd /content/GroupProject && python train_expert.py Ant-v4 10000000 1 norm tuned_ant
# resume:
# !cd /content/GroupProject && python train_expert.py Ant-v4 10000000 1 norm tuned_ant resume
# Faster fallback (our config, ~8h, reached ~2850): 
# !cd /content/GroupProject && python train_expert.py Ant-v4 16000000 8 norm

## 6. Stage 2 - demonstration collection + EDA + quality gate (M2)

In [ ]:
!cd /content/GroupProject && python collect_demos.py Walker2d-v4 100
!cd /content/GroupProject && python collect_demos.py Ant-v4 100

## 7. Stage 3 - BC experiments (M3, M4, M5) and multi-seed arch sweep (M8)

Use a GPU runtime here for the from-scratch BC runs.

In [ ]:
!cd /content/GroupProject && python bc_experiments.py Walker2d-v4
!cd /content/GroupProject && python arch_sweep.py Walker2d-v4
# repeat for Ant once its expert and dataset are ready:
# !cd /content/GroupProject && python bc_experiments.py Ant-v4
# !cd /content/GroupProject && python arch_sweep.py Ant-v4

## 8. Stages 4-5 - DAgger and pretraining

Run `notebooks/04_dagger.ipynb` and `notebooks/05_pretraining.ipynb` (still in development). All outputs already persist to Drive via `PROJECT_DATA_ROOT`.

## 9. Bonus - SAC off-policy experts (vs on-policy PPO)

SAC is update-bound (a gradient step per env step), so unlike PPO it is much faster on a GPU: select Runtime -> Change runtime type -> GPU. 3M steps is roughly 1-2h per env on a Colab GPU. No VecNormalize (off-policy replay buffer), so the SAC expert produces raw-observation demonstrations. If a run is interrupted, re-run the same cell with `resume` appended: it reloads the latest checkpoint from Drive and continues (the policy resumes; the replay buffer restarts). best_model lands in `models/sac_expert_<env>/` on Drive.

In [ ]:
# Ant-v4: in-brief comparison against the PPO expert (target 4000; SAC realistic ~5000-7000).
!cd /content/GroupProject && python train_sac.py Ant-v4 3000000 tuned_ant
# resume after a disconnect:
# !cd /content/GroupProject && python train_sac.py Ant-v4 3000000 tuned_ant resume

In [ ]:
# HalfCheetah-v4: off-brief, the clean path past 8000 (SAC ~9000-11000).
!cd /content/GroupProject && python train_sac.py HalfCheetah-v4 3000000 tuned_halfcheetah
# resume:
# !cd /content/GroupProject && python train_sac.py HalfCheetah-v4 3000000 tuned_halfcheetah resume

## Recovering after a disconnect

Models, checkpoints (every 100k steps), datasets, and figures live under `MyDrive/rl_project/`. Re-run cells 1-4 to remount and reattach, then continue. Note: `train_expert.py` restarts a run from scratch rather than resuming mid-training, so prefer Colab Pro or keep the tab active for the long expert runs.